# 第15章 模板匹配（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 15 章跟随代码。**全量收录原书代码**（例 15.1~例 15.3 + 章内独立演示），缺失例题已按书补全，代码以网页笔记 docs/opencv/ch15.md 为准（含笔误与参数修正）。
> `cv2.imshow` 需在**本地 Python 环境**运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 `imread` 返回 None。
> 素材替换：原书 lena512g.bmp / temp.bmp 用 picture3.jpg（代码中读为灰度图）+ picture3_mouth.jpg 顶替；多模板匹配的原书 lena4.bmp（由 4 幅相同图像构成）用 `np.tile` 把 picture3 平铺成 2×2 模拟，代码结构不变。

# 第十五章 模板匹配

> 模板匹配（template matching）在输入图像 A 内寻找与模板图像 B 最相似的部分：把模板图像在输入图像上逐像素滑动，每到一个位置用一种相似度度量打一个分，全部位置组成一张"分数地图"——分数地图上最暗处（平方差类方法）或最亮处（相关类方法）就是模板所在的位置。全章例题：例 15.1~15.2（单目标匹配与定位标记）、例 15.3（np.where + 阈值的多目标匹配）。

## 15.1 模板匹配基础

模板匹配是指在当前图像 A 内寻找与图像 B 最相似的部分。一般将图像 A 称为**输入图像**（input image），将图像 B 称为**模板图像**（template image）。操作方法是将模板图像 B 在图像 A 上滑动，遍历所有像素以完成匹配。OpenCV 用 `cv2.matchTemplate()` 实现：

```text
result = cv2.matchTemplate( image, templ, method[, mask] )
```

- **image**：原始图像（输入图像），必须是 8 位或者 32 位的浮点型图像；
- **templ**：模板图像，尺寸必须小于或等于原始图像，并且与原始图像具有同样的类型；
- **method**：匹配方法，有 6 种取值（对应数值 0~5）：
    - `cv2.TM_SQDIFF`（0）：平方差匹配，完全匹配则结果为零，值越大匹配越差 → 最佳位置在**最小值**处；
    - `cv2.TM_SQDIFF_NORMED`（1）：归一化平方差匹配 → 最小值处；
    - `cv2.TM_CCORR`（2）：相关匹配，模板与对应位置相乘累加，乘积越大匹配越好 → 最大值处；
    - `cv2.TM_CCORR_NORMED`（3）：归一化相关匹配 → 最大值处；
    - `cv2.TM_CCOEFF`（4）：相关系数匹配（先减去各自均值再相关）→ 最大值处；
    - `cv2.TM_CCOEFF_NORMED`（5）：归一化相关系数匹配，值域 [-1, 1]，1 表示完美匹配 → 最大值处；
- **mask**：模板图像掩模，与模板同类型、同大小，通常使用默认值。

**返回值 result** 是每个位置的比较结果组合构成的一个结果集（"分数地图"），类型是**单通道 32 位浮点型**——注意它不是"最匹配区域的位置"，每个元素只是一个位置的匹配分数。输入图像 W×H、模板 w×h 时，result 的尺寸为 (W−w+1)×(H−h+1)；numpy 先行后列，`res.shape` 是 (H−h+1, W−w+1)。

**定位三行式**（配合 `cv2.minMaxLoc()` 与 `cv2.rectangle()` 标记匹配位置）：

```python
minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(rv)   # 一次给出最值与位置，位置是 (列, 行)
topLeft = minLoc                                     # SQDIFF 类取 minLoc，其余四类取 maxLoc
bottomRight = (topLeft[0] + w, topLeft[1] + h)       # 左上角加模板宽高得右下角
cv2.rectangle(img, topLeft, bottomRight, 255, 2)     # 白色矩形圈出匹配区域
```

> **勘误**：表 15-1 里"1 表示完美匹配、-1 表示糟糕的匹配、0 表示没有相关性"这组值域，只对归一化的 TM_CCOEFF_NORMED 严格成立；TM_CCOEFF 未归一化，实测分数可达 4.7×10⁷。实际项目筛选匹配结果，优先用 `_NORMED` 系列。

**动手验证**（笔记补充）：构造一幅 300×400 的已知大图，从第 87 行 123 列裁出 60 行 80 列的模板，六种方法的最佳位置必须一致且指回裁剪处 (123, 87)。同时验证：result 的形状 (241, 321) 正是 (300−60+1, 400−80+1)；打印出来的方法数值 0~5 与表 15-1 的"对应数值"一列一致。

In [ ]:
# ===== 15.1 动手验证：已知图 + 从其中裁出的模板，六种方法位置对照 =====
import cv2
import numpy as np

rng = np.random.RandomState(42)
big = rng.randint(0, 256, (300, 400)).astype(np.uint8)   # 300 行 400 列的已知大图
templ = big[87:147, 123:203].copy()                      # 从第 87 行 123 列裁出 60 行 80 列模板

methods = [                                    # (方法, 判定方向)：平方差类取 min，其余取 max
    (cv2.TM_SQDIFF, 'min'),
    (cv2.TM_SQDIFF_NORMED, 'min'),
    (cv2.TM_CCORR, 'max'),
    (cv2.TM_CCORR_NORMED, 'max'),
    (cv2.TM_CCOEFF, 'max'),
    (cv2.TM_CCOEFF_NORMED, 'max'),
]
for m, mode in methods:                                  # 逐个方法试验
    res = cv2.matchTemplate(big, templ, m)               # 模板匹配
    minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(res)  # 找最值及位置
    best = minLoc if mode == 'min' else maxLoc           # 按方向取最佳位置
    print(m, 'res.shape=', res.shape, '最佳位置=', best)   # m 打印出来是方法的枚举数值


### 例 15.1 · TM_SQDIFF：找最小值

method 取 cv2.TM_SQDIFF（平方差）：值越小越匹配，0 表示完全匹配，所以最佳位置在**最小值**处（minLoc）。左图画出 matchTemplate 的返回值（分数地图），最暗的一点就是最佳匹配位置；右图在输入图像上用白色矩形圈出嘴部。实测 rv.shape=(993, 503)、minVal=805696.0、topLeft=(273, 603)。

In [ ]:
# ===== 15.1 例 15.1：TM_SQDIFF 模板匹配（原书 lena512g.bmp + temp.bmp → picture3.jpg + picture3_mouth.jpg） =====
import cv2
import numpy as np
from matplotlib import pyplot as plt

img = cv2.imread('picture3.jpg', 0)               # 输入图像，flags=0 读成灰度图
template = cv2.imread('picture3_mouth.jpg', 0)    # 模板图像：从输入图中截出的嘴部子图
th, tw = template.shape[::]                       # shape 正序取：th 是高，tw 是宽

rv = cv2.matchTemplate(img, template, cv2.TM_SQDIFF)    # 平方差匹配：值越小越匹配
minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(rv)      # 找最值及位置
topLeft = minLoc                                        # SQDIFF 取最小值位置为左上角
bottomRight = (topLeft[0] + tw, topLeft[1] + th)        # 左上角加模板宽高得右下角
cv2.rectangle(img, topLeft, bottomRight, 255, 2)        # 白色矩形圈出匹配区域

plt.subplot(121), plt.imshow(rv, cmap='gray')           # 左图：匹配分数地图
plt.title('Matching Result'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(img, cmap='gray')          # 右图：标记后的输入图像
plt.title('Detected Point'), plt.xticks([]), plt.yticks([])
plt.show()


### shape[::] 与 shape[::-1]

计算模板宽度、高度时，例 15.1 使用 `th, tw = template.shape[::]`，而 OpenCV 官网示例常用 `tw, th = template.shape[::-1]`。`template.shape` 永远是 (高, 宽)（行在前）：`shape[::]` 正序取，先高后宽；`shape[::-1]` 倒序取，先宽后高——两种写法都行，关键是**变量顺序要跟写法配套**，配错了矩形就画偏。

In [ ]:
# ===== 15.1 shape[::] 与 shape[::-1]：正序先高后宽，倒序先宽后高 =====
import numpy as np

list2d = np.arange(18).reshape(3, 6)   # 3 行 6 列数组，相当于高 3、宽 6 的"模板"
print(list2d)

h, w = list2d.shape[::]                # 正序取 shape：第 1 个是高度，第 2 个是宽度
print(h, w)

w, h = list2d.shape[::-1]              # 倒序取 shape：第 1 个是宽度，第 2 个是高度
print(w, h)


### 例 15.2 · TM_CCOEFF：找最大值

method 换成 cv2.TM_CCOEFF（相关系数）：返回值越大匹配度越好，最佳位置在**最大值**处（maxLoc）。同样的输入图像和模板图像，实测 maxVal=46961540.0、topLeft=(273, 603)，与例 15.1 圈出的是同一块区域。查找最佳匹配前，先确定用的是什么 method，再确定找最大值还是最小值。

In [ ]:
# ===== 15.1 例 15.2：TM_CCOEFF 模板匹配（原书 lena512g.bmp + temp.bmp → picture3.jpg + picture3_mouth.jpg） =====
import cv2
import numpy as np
from matplotlib import pyplot as plt

img = cv2.imread('picture3.jpg', 0)               # 输入图像，灰度
template = cv2.imread('picture3_mouth.jpg', 0)    # 模板图像
tw, th = template.shape[::-1]                     # 倒序取：tw 是宽，th 是高

rv = cv2.matchTemplate(img, template, cv2.TM_CCOEFF)    # 相关系数匹配：值越大越匹配
minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(rv)
topLeft = maxLoc                                        # CCOEFF 取最大值位置为左上角
bottomRight = (topLeft[0] + tw, topLeft[1] + th)
cv2.rectangle(img, topLeft, bottomRight, 255, 2)

plt.subplot(121), plt.imshow(rv, cmap='gray')
plt.title('Matching Result'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(img, cmap='gray')
plt.title('Detected Point'), plt.xticks([]), plt.yticks([])
plt.show()


## 15.2 多模板匹配

有些情况下，要搜索的模板图像很可能在输入图像内出现**多次**，这时需要找出多个匹配结果。而 `cv2.minMaxLoc()` 仅仅能够找出最值，无法给出所有匹配区域的位置信息。要匹配多个结果，需要利用**阈值**（threshold）进行处理，分五步完成：

1. **np.where 筛位置**：`loc = np.where(res >= threshold)` 获取满足条件的位置集合（返回行索引组、列索引组两个数组）；
2. **循环处理**：for 循环逐个取出每个位置；
3. **zip(*loc) 配对**：`zip(*t)` 把行、列索引组按位打包成一个个元组；
4. **loc[::-1] 行列互换**：rectangle 的顶点参数是 (列, 行)，先 `loc[::-1]` 倒序再 zip 配对；
5. **逐个画矩形**：`cv2.rectangle(img, pt, (pt[0]+w, pt[1]+h), 255, 2)` 标记每个命中位置。

**第 1 步 · np.where 获取满足条件的位置集合**：一维数组返回一组索引数组；二维数组返回两组索引数组（行索引组、列索引组）。套到模板匹配上，就是找出返回值 res 中所有不小于阈值的位置：`loc = np.where(res >= threshold)`。

In [ ]:
# ===== 15.2 np.where：一维数组，找大于 5 的元素索引 =====
import numpy as np

a = np.array([3, 6, 8, 1, 2, 88])
b = np.where(a > 5)              # 条件：元素值大于 5（索引从 0 开始）
print(b)


In [ ]:
# ===== 15.2 np.where：二维数组，返回行、列两组索引，再用 zip 逐个取出 =====
import numpy as np

am = np.array([[3, 6, 8, 77, 66], [1, 2, 88, 3, 98], [11, 2, 67, 5, 2]])
b = np.where(am > 5)
print(b)
for i in zip(*b):                # 逐个取出 (行号, 列号)
    print(i)


**第 2 步 · 用循环处理多个位置**：获取匹配值的索引集合后，就能遍历所有匹配的位置并逐个标记——处理多个值，通常需要循环。

In [ ]:
# ===== 15.2 循环：逐个处理列表里的值 =====
value = [71, 23, 16]
for i in value:
    print('value 内的值:', i)


**第 3 步 · 在循环中使用 zip(*loc)**：`np.where()` 返回的是"行索引组、列索引组"两个独立数组，循环前要把它们按位配对。`zip()` 将可迭代对象中对应的元素打包成一个个元组，`*t` 先解包再按位打包，于是遍历语句写作 `for i in zip(*模板匹配索引集合)`，每次得到的 i 就是一个 (行号, 列号) 元组。

In [ ]:
# ===== 15.2 zip(*t)：把三个列表按位打包 =====
x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]
t = (x, y, z)          # 元组里装三个列表
print(t)
for i in zip(*t):      # *t 解包成三个参数，zip 按位打包
    print(i)


**第 4 步 · 调整坐标：行列互换**：`np.where()` 返回的位置形式是 (行号, 列号)，而 `cv2.rectangle()` 的顶点参数用的是 (列号, 行号) 形式。画矩形前要做"行列互换"，一条倒序切片 `loc[::-1]` 即可。所以遍历语句最终写作 `for pt in zip(*loc[::-1])`：先换成列在前，再 zip 配对，每次得到的 pt 就是 rectangle 需要的 (列, 行) 顶点。

In [ ]:
# ===== 15.2 loc[::-1]：把 (行, 列) 交换成 (列, 行) =====
import numpy as np

loc = ([1, 2, 3, 4], [11, 12, 13, 14])   # 模拟 where 的返回：行索引组在前
print(loc)
print(loc[::-1])                          # 倒序：列索引组换到前面


**第 5 步 · 标记匹配位置**：对每个命中位置 pt，以 pt 为左上角、pt 加模板宽高为右下角画矩形：

```python
for pt in zip(*loc[::-1]):
    cv2.rectangle(img, pt, (pt[0] + w, pt[1] + h), 255, 2)   # pt 是 (列, 行) 顶点
```

> **勘误**：原书 15.2 节的步骤编号有两个"4."——"4. 调整坐标"之后又出现"4. 标记匹配图像的位置"，后者应为"5."，本笔记按 5 步组织。

### 例 15.3 · 多目标匹配

原书输入图像 lena4.bmp 由 4 幅 lena 图像构成；这里用 `np.tile(img, (2, 2))` 把 picture3 平铺成 2×2，构造同样的场景。用 TM_CCOEFF_NORMED 匹配 + 阈值 0.9 筛选，实测命中 216 个位置、聚集在 4 个目标附近（左上、右上、左下、右下各一个），全部被圈出。

标记出来的边框远粗于设定的线宽 1，是因为当前区域内存在多个大于阈值的位置，每一个位置都画了一个宽度为 1 的矩形，多个矩形叠在一起边界就显得很粗——调整阈值可以改变效果（实测阈值提到 0.99 后恰好剩 4 个命中点）。

In [ ]:
# ===== 15.2 例 15.3：多模板匹配（原书 lena4.bmp + lena4Temp.bmp → 平铺 2×2 的 picture3.jpg + picture3_mouth.jpg） =====
import cv2
import numpy as np
from matplotlib import pyplot as plt

img = cv2.imread('picture3.jpg', 0)              # 灰度读入
img = np.tile(img, (2, 2))                       # 行、列各重复一次：4 个相同子图，模拟 lena4
template = cv2.imread('picture3_mouth.jpg', 0)   # 模板图像
w, h = template.shape[::-1]                      # 倒序取 shape：先宽后高

res = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)   # 归一化相关系数：越大越好
threshold = 0.9                                  # 阈值：分数不低于 0.9 的都算命中
loc = np.where(res >= threshold)                 # 命中位置的 (行索引组, 列索引组)
for pt in zip(*loc[::-1]):                       # 行列互换后逐个取出 (列, 行)
    cv2.rectangle(img, pt, (pt[0] + w, pt[1] + h), 255, 1)     # 每个命中点画一个矩形

plt.imshow(img, cmap='gray')                     # 显示标记结果
plt.xticks([]), plt.yticks([])
plt.show()


**拓展演示**（笔记补充）：带掩模的模板匹配——mask 与模板同类型、同大小，掩模中为 0 的位置不参与匹配计算，为非 0 的位置正常参与。原书（与 OpenCV 文档）声明该参数当前仅支持 TM_SQDIFF 和 TM_CCORR_NORMED；实测 OpenCV 4.13 中六种方法传入 mask 均不报错，但**未归一化方法配掩模不可靠**，归一化方法（如 TM_CCORR_NORMED）可靠。下面把模板左侧 1/3 掩掉，实测最佳位置仍是 (273, 603)。

In [ ]:
# ===== 15.1 拓展：带掩模的模板匹配（掩模置 0 的部分不参与比较） =====
import cv2
import numpy as np

img = cv2.imread('picture3.jpg', 0)
template = cv2.imread('picture3_mouth.jpg', 0)
h, w = template.shape
mask = np.full((h, w), 255, dtype=np.uint8)   # 全 255：先让所有像素参与比较
mask[:, :w // 3] = 0                          # 左侧 1/3 置 0：这部分不参与比较

res = cv2.matchTemplate(img, template, cv2.TM_CCORR_NORMED, mask=mask)
minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(res)
print(maxLoc, maxVal)                         # 实测 (273, 603) 0.9994
